# Verification notebook — *Representing EEG Functional Connectivity as a FAIR Knowledge Graph*

This notebook independently reproduces every numerical result reported in the paper directly from the four TriG knowledge graph files in `../kg/`. No EDF source files are required.

**Structure**
1. Setup and KG loading
2. Table 7 — KG characterisation (triple counts, named graphs, predicates)
3. Table 9 — SPARQL analytical queries Q1–Q5
4. PROV-O completeness (6/6)
5. SHACL structural validation (0 violations)
6. CQ evaluation (12/12)
7. Table 11 — FAIR compliance (15/15)
8. Summary comparison table

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from rdflib import ConjunctiveGraph
import pyshacl

KG_DIR = Path('../kg')
KG_FILES = {
    'chb01_nonsz':   KG_DIR / 'chb01_nonsz.trig',
    'chb01_seizure': KG_DIR / 'chb01_seizure.trig',
    'S001_rest':     KG_DIR / 'S001_rest.trig',
    'S001_task':     KG_DIR / 'S001_task.trig',
}

print('Loading KGs...')
kgs = {}
for name, path in KG_FILES.items():
    cg = ConjunctiveGraph()
    cg.parse(path, format='trig')
    kgs[name] = cg
    print(f'  {name}: {len(list(cg.quads()))} triples')
print('Done.')

## 1. Table 7 — KG characterisation

Expected values (paper Table 7):

| KG | Triples | Named graphs | Predicates |
|---|---|---|---|
| chb01_nonsz | 9,590 | 12 | 31 |
| chb01_seizure | 11,570 | 12 | 31 |
| S001_rest | 38,493 | 8 | 26 |
| S001_task | 116,013 | 16 | 26 |

In [ ]:
EXPECTED = {
    'chb01_nonsz':   {'triples': 9590,   'named_graphs': 12, 'predicates': 31},
    'chb01_seizure': {'triples': 11570,  'named_graphs': 12, 'predicates': 31},
    'S001_rest':     {'triples': 38493,  'named_graphs': 8,  'predicates': 26},
    'S001_task':     {'triples': 116013, 'named_graphs': 16, 'predicates': 26},
}

print(f"{'KG':<18} {'Triples':>8} {'Expected':>8} {'Graphs':>7} {'Expected':>8} {'Preds':>6} {'Expected':>8} {'OK?':>5}")
print('-' * 75)
for name, cg in kgs.items():
    quads  = list(cg.quads())
    t      = len(quads)
    graphs = len({g for s,p,o,g in quads})
    preds  = len({p for s,p,o,g in quads})
    exp    = EXPECTED[name]
    ok     = (t == exp['triples'] and graphs == exp['named_graphs'] and preds == exp['predicates'])
    print(f"{name:<18} {t:>8,} {exp['triples']:>8,} {graphs:>7} {exp['named_graphs']:>8} {preds:>6} {exp['predicates']:>8}  {'✓' if ok else '✗ MISMATCH'}")

## 2. Table 9 — SPARQL Queries Q1–Q5

Queries are loaded from `../src/kg_experiments.py`.

In [ ]:
from src.kg_experiments import run_sparql_queries

print('Running Q1–Q5 on all four KGs...\n')
results = {}
for name, cg in kgs.items():
    r = run_sparql_queries(cg)
    results[name] = r
    print(f'--- {name} ---')
    print(f"  Q1 mapped electrodes : {r.get('Q1_nifstd_mapped')}")
    print(f"  Q2 top-5 edges       : {r.get('Q2_top5_edges', [])[:2]} ...")
    print(f"  Q3 windows with edges: {len(r.get('Q3_edges_per_window', {}))}")
    print(f"  Q4 agent / version   : {r.get('Q4_agent_name')} {r.get('Q4_agent_version')}")
    print(f"  Q4 measure           : {r.get('Q4_measure')}")
    print(f"  Q5 pole sharing      : {r.get('Q5_pole_sharing')}")
    print()

print('Expected Q5: True for CHB-MIT, False for EEGMMIDB')
print(f"  chb01_nonsz   Q5 = {results['chb01_nonsz']['Q5_pole_sharing']}  (expected True)")
print(f"  chb01_seizure Q5 = {results['chb01_seizure']['Q5_pole_sharing']}  (expected True)")
print(f"  S001_rest     Q5 = {results['S001_rest']['Q5_pole_sharing']}  (expected False)")
print(f"  S001_task     Q5 = {results['S001_task']['Q5_pole_sharing']}  (expected False)")

## 3. PROV-O completeness

Expected: 6/6 checks pass for all four KGs.

In [ ]:
from src.kg_experiments import check_prov_completeness

for name, cg in kgs.items():
    checks = check_prov_completeness(cg)
    passed = sum(checks.values())
    total  = len(checks)
    print(f'{name}: {passed}/{total}  {"✓" if passed == total else "✗"}')
    for k, v in checks.items():
        print(f'  {k}: {"✓" if v else "✗"}')

## 4. SHACL structural validation

Expected: 0 violations for all four KGs.

In [ ]:
from rdflib import Graph

shapes = Graph()
shapes.parse('../shacl/eego-shapes.ttl', format='turtle')

for name, cg in kgs.items():
    data_g = Graph()
    for s, p, o in cg:
        data_g.add((s, p, o))
    conforms, _, report_text = pyshacl.validate(
        data_g, shacl_graph=shapes, abort_on_first=False
    )
    violations = report_text.count('sh:Violation')
    print(f'{name}: conforms={conforms}, violations={violations}  {"✓" if conforms else "✗"}')

## 5. CQ evaluation

Expected: 12/12 competency questions answered.

In [ ]:
with open('../outputs/cq_evaluation.json') as f:
    cq = json.load(f)

print(f'CQ results from pipeline run:')
for k, v in cq.items():
    print(f'  {k}: {v}')

## 6. FAIR evaluation

Expected: 15/15 sub-criteria satisfied.

In [ ]:
from src.kg_experiments import fair_evaluation

# Run on the merged dataset
merged = ConjunctiveGraph()
for cg in kgs.values():
    for q in cg.quads():
        merged.add(q)

fair = fair_evaluation(merged)
score = fair.pop('score')
print(f'FAIR score: {score}')
for k, v in fair.items():
    print(f'  {k}: {"✓" if v else "✗"}')

## 7. Summary comparison table

In [ ]:
print('='*65)
print('SUMMARY: Paper values vs. reproduced values')
print('='*65)

rows = [
    ('chb01_nonsz triples',    9590,   len(list(kgs['chb01_nonsz'].quads()))),
    ('chb01_seizure triples',  11570,  len(list(kgs['chb01_seizure'].quads()))),
    ('S001_rest triples',      38493,  len(list(kgs['S001_rest'].quads()))),
    ('S001_task triples',      116013, len(list(kgs['S001_task'].quads()))),
    ('Total triples',          175666, sum(len(list(cg.quads())) for cg in kgs.values())),
    ('Q1 CHB-MIT mapped',      21,     results['chb01_nonsz']['Q1_nifstd_mapped']),
    ('Q1 EEGMMIDB mapped',     64,     results['S001_rest']['Q1_nifstd_mapped']),
    ('Q5 CHB-MIT pole share',  True,   results['chb01_nonsz']['Q5_pole_sharing']),
    ('Q5 EEGMMIDB pole share', False,  results['S001_rest']['Q5_pole_sharing']),
]

print(f"{'Metric':<30} {'Paper':>10} {'Reproduced':>12} {'Match':>6}")
print('-'*62)
for label, expected, actual in rows:
    match = '✓' if expected == actual else '✗'
    print(f"{label:<30} {str(expected):>10} {str(actual):>12} {match:>6}")